# Modeling

This notebook trains and evaluates baseline and machine learning models for predicting season long full-PPR fantasy points.

Models are evaluated using time based validation so that each validation season is predicted only from earlier seasons.

In [1]:
import polars as pl

df = pl.read_csv("../data/processed/modeling_dataset_2018_2025.csv")

print(df.shape)
df.group_by("season").len().sort("season")

(2909, 76)


season,len
i64,u32
2018,307
2019,317
2020,362
2021,389
2022,394
2023,374
2024,368
2025,398


## Baseline

Baseline model just using last season's fantasy points to predict this season

In [2]:
baseline_df = df.filter(
    pl.col("fantasy_points_lag_1").is_not_null()
).with_columns(
    pl.col("fantasy_points_lag_1")
      .alias("baseline_prediction")
)

In [3]:
baseline_metrics = baseline_df.select([
    (
        pl.col("fantasy_points_ppr_calc")
        - pl.col("baseline_prediction")
    )
    .abs()
    .mean()
    .alias("mae"),

    (
        (
            pl.col("fantasy_points_ppr_calc")
            - pl.col("baseline_prediction")
        ) ** 2
    )
    .mean()
    .sqrt()
    .alias("rmse")
])

baseline_metrics

mae,rmse
f64,f64
47.957016,68.482623


In [4]:
baseline_by_position = (
    baseline_df
    .group_by("position")
    .agg([
        (
            pl.col("fantasy_points_ppr_calc")
            - pl.col("baseline_prediction")
        )
        .abs()
        .mean()
        .alias("mae"),

        (
            (
                pl.col("fantasy_points_ppr_calc")
                - pl.col("baseline_prediction")
            ) ** 2
        )
        .mean()
        .sqrt()
        .alias("rmse")
    ])
    .sort("position")
)

baseline_by_position

position,mae,rmse
str,f64,f64
"""QB""",64.99386,92.610864
"""RB""",53.239807,74.687359
"""TE""",32.863969,45.634756
"""WR""",46.546404,64.023121


70 / 30 Weighted Baseline from EDA

In [5]:
weighted_baseline_df = df.filter(
    pl.col("fantasy_points_2yr_weighted").is_not_null()
).with_columns(
    pl.col("fantasy_points_2yr_weighted")
      .alias("baseline_prediction")
)

In [6]:
weighted_baseline_metrics = weighted_baseline_df.select([
    (
        pl.col("fantasy_points_ppr_calc")
        - pl.col("baseline_prediction")
    )
    .abs()
    .mean()
    .alias("mae"),

    (
        (
            pl.col("fantasy_points_ppr_calc")
            - pl.col("baseline_prediction")
        ) ** 2
    )
    .mean()
    .sqrt()
    .alias("rmse")
])

weighted_baseline_metrics

mae,rmse
f64,f64
46.043621,64.551699


In [7]:
weighted_baseline_by_position = (
    weighted_baseline_df
    .group_by("position")
    .agg([
        (
            pl.col("fantasy_points_ppr_calc")
            - pl.col("baseline_prediction")
        )
        .abs()
        .mean()
        .alias("mae"),

        (
            (
                pl.col("fantasy_points_ppr_calc")
                - pl.col("baseline_prediction")
            ) ** 2
        )
        .mean()
        .sqrt()
        .alias("rmse")
    ])
    .sort("position")
)

weighted_baseline_by_position

position,mae,rmse
str,f64,f64
"""QB""",60.485065,84.773374
"""RB""",51.863113,71.442984
"""TE""",31.186835,43.249383
"""WR""",45.14853,60.838066


In [8]:
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [9]:
model_data = df.to_pandas()

# RB only model to start

In [10]:
rb_features = [
    "age",
    "age_squared",
    "years_exp",
    "draft_number_filled",
    "undrafted_flag",
    "team_change_flag",

    "fantasy_points_lag_1",
    "fantasy_points_lag_2",
    "fantasy_ppg_lag_1",
    "fantasy_points_2yr_weighted",

    "games_lag_1",
    "games_2yr_avg",

    "fantasy_points_change",

    "targets_lag_1",
    "carries_lag_1",
    "receptions_lag_1",
    "opportunities_lag_1",

    "targets_per_game_lag_1",
    "carries_per_game_lag_1",
    "opportunities_per_game_lag_1",

    "yards_per_carry_lag_1",
    "fantasy_points_per_opportunity_lag_1",

    "targets_2yr_avg",
    "carries_2yr_avg",
    "opportunities_2yr_avg"
]

target = "fantasy_points_ppr_calc"

In [11]:
rb_df = model_data[
    model_data["position"] == "RB"
].copy()

In [12]:
validation_seasons = [2021, 2022, 2023, 2024, 2025]

In [13]:
rb_linear_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression())
])

In [14]:
rb_results = []

for validation_season in validation_seasons:

    train = rb_df[
        rb_df["season"] < validation_season
    ]

    valid = rb_df[
        rb_df["season"] == validation_season
    ]

    X_train = train[rb_features]
    y_train = train[target]

    X_valid = valid[rb_features]
    y_valid = valid[target]

    rb_linear_model.fit(X_train, y_train)

    predictions = rb_linear_model.predict(X_valid)

    mae = mean_absolute_error(
        y_valid,
        predictions
    )

    rmse = mean_squared_error(
        y_valid,
        predictions
    ) ** 0.5

    rb_results.append({
        "season": validation_season,
        "train_rows": len(train),
        "validation_rows": len(valid),
        "mae": mae,
        "rmse": rmse
    })

In [15]:
rb_results_df = pd.DataFrame(rb_results)

rb_results_df

,season,train_rows,validation_rows,mae,rmse
0,2021,250,99,44.517064,58.599303
1,2022,349,100,48.096662,65.477522
2,2023,449,93,45.666090,59.796570
3,2024,542,89,49.025673,67.242573
4,2025,631,95,48.735904,66.178768


In [16]:
rb_results_df[
    ["mae", "rmse"]
].mean()

mae     47.208279
rmse    63.458947
dtype: float64

In [17]:
rb_baseline_results = []

for validation_season in validation_seasons:

    valid = rb_df[
        rb_df["season"] == validation_season
    ].copy()

    y_valid = valid[target]

    baseline_predictions = valid[
        "fantasy_points_2yr_weighted"
    ]

    mae = mean_absolute_error(
        y_valid,
        baseline_predictions
    )

    rmse = mean_squared_error(
        y_valid,
        baseline_predictions
    ) ** 0.5

    rb_baseline_results.append({
        "season": validation_season,
        "baseline_mae": mae,
        "baseline_rmse": rmse
    })

rb_baseline_results_df = pd.DataFrame(
    rb_baseline_results
)

rb_baseline_results_df

,season,baseline_mae,baseline_rmse
0,2021,52.160929,67.150246
1,2022,49.155720,68.551232
2,2023,49.609785,71.158244
3,2024,46.841056,68.120926
4,2025,48.621789,68.609732


In [18]:
rb_comparison = rb_results_df.merge(
    rb_baseline_results_df,
    on="season"
)

rb_comparison

,season,train_rows,validation_rows,mae,rmse,baseline_mae,baseline_rmse
0,2021,250,99,44.517064,58.599303,52.160929,67.150246
1,2022,349,100,48.096662,65.477522,49.155720,68.551232
2,2023,449,93,45.666090,59.796570,49.609785,71.158244
3,2024,542,89,49.025673,67.242573,46.841056,68.120926
4,2025,631,95,48.735904,66.178768,48.621789,68.609732


In [19]:
rb_comparison["mae_improvement"] = (
    rb_comparison["baseline_mae"]
    - rb_comparison["mae"]
)

rb_comparison["rmse_improvement"] = (
    rb_comparison["baseline_rmse"]
    - rb_comparison["rmse"]
)

rb_comparison

,season,train_rows,validation_rows,mae,rmse,baseline_mae,baseline_rmse,mae_improvement,rmse_improvement
0,2021,250,99,44.517064,58.599303,52.160929,67.150246,7.643866,8.550943
1,2022,349,100,48.096662,65.477522,49.155720,68.551232,1.059058,3.073710
2,2023,449,93,45.666090,59.796570,49.609785,71.158244,3.943695,11.361674
3,2024,542,89,49.025673,67.242573,46.841056,68.120926,-2.184617,0.878354
4,2025,631,95,48.735904,66.178768,48.621789,68.609732,-0.114115,2.430963


In [20]:
rb_comparison[[
    "mae",
    "baseline_mae",
    "mae_improvement",
    "rmse",
    "baseline_rmse",
    "rmse_improvement"
]].mean()

mae                 47.208279
baseline_mae        49.277856
mae_improvement      2.069577
rmse                63.458947
baseline_rmse       68.718076
rmse_improvement     5.259129
dtype: float64

# Refactor RB walk-forward validation into a reusable function

In [26]:
def run_walk_forward_linear_model(
    data,
    position,
    features,
    target,
    validation_seasons
):
    """
    Evaluate a position-specific linear regression model using
    walk-forward validation.

    For each validation season, the model trains only on earlier seasons.
    """

    position_data = data[
        data["position"] == position
    ].copy()

    fold_results = []

    for validation_season in validation_seasons:

        train = position_data[
            position_data["season"] < validation_season
        ].copy()

        valid = position_data[
            position_data["season"] == validation_season
        ].copy()

        assert train["season"].max() < validation_season
        assert valid["season"].nunique() == 1
        assert valid["season"].iloc[0] == validation_season

        if train.empty or valid.empty:
            continue

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LinearRegression())
        ])

        model.fit(
            train[features],
            train[target]
        )

        predictions = model.predict(valid[features])

        fold_results.append({
            "position": position,
            "validation_season": validation_season,
            "train_rows": len(train),
            "validation_rows": len(valid),
            "mae": mean_absolute_error(
                valid[target],
                predictions
            ),
            "rmse": mean_squared_error(
                valid[target],
                predictions
            ) ** 0.5
        })

    fold_results = pd.DataFrame(fold_results)

    summary = pd.DataFrame([{
        "position": position,
        "mean_mae": fold_results["mae"].mean(),
        "mean_rmse": fold_results["rmse"].mean(),
        "validation_folds": len(fold_results)
    }])

    return fold_results, summary

In [27]:
rb_fold_results, rb_summary = run_walk_forward_linear_model(
    data=model_data,
    position="RB",
    features=rb_features,
    target=target,
    validation_seasons=validation_seasons
)

display(rb_fold_results)
display(rb_summary)

,position,validation_season,train_rows,validation_rows,mae,rmse
0,RB,2021,250,99,44.517064,58.599303
1,RB,2022,349,100,48.096662,65.477522
2,RB,2023,449,93,45.666090,59.796570
3,RB,2024,542,89,49.025673,67.242573
4,RB,2025,631,95,48.735904,66.178768


,position,mean_mae,mean_rmse,validation_folds
0,RB,47.208279,63.458947,5


### Walk-forward validation refactor

Converted the RB linear-regression evaluation into a reusable function.

Validation design:
- Position-specific models
- Training data limited to seasons before the validation season
- Validation seasons: 2021–2025
- Median imputation performed inside the modeling pipeline

Regression check:
- Previous RB MAE: 47.21
- Refactored RB MAE: 47.21
- Previous RB RMSE: 63.46
- Refactored RB RMSE: 63.46

Next step: define the appropriate feature lists and run the same framework
for QB, WR, and TE.

# Run it for each position

There's meaningful improvement from baseline to regression so now time to apply same method to all positions

In [21]:
def evaluate_linear_model(data, features, target, validation_seasons):
    results = []

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LinearRegression())
    ])

    for validation_season in validation_seasons:

        train = data[data["season"] < validation_season]
        valid = data[data["season"] == validation_season]

        X_train = train[features]
        y_train = train[target]

        X_valid = valid[features]
        y_valid = valid[target]

        model.fit(X_train, y_train)

        predictions = model.predict(X_valid)

        mae = mean_absolute_error(y_valid, predictions)
        rmse = mean_squared_error(y_valid, predictions) ** 0.5

        baseline_predictions = valid["fantasy_points_2yr_weighted"]

        baseline_mae = mean_absolute_error(
            y_valid,
            baseline_predictions
        )

        baseline_rmse = mean_squared_error(
            y_valid,
            baseline_predictions
        ) ** 0.5

        results.append({
            "season": validation_season,
            "mae": mae,
            "rmse": rmse,
            "baseline_mae": baseline_mae,
            "baseline_rmse": baseline_rmse
        })

    return pd.DataFrame(results)

In [22]:
qb_df = model_data[model_data["position"] == "QB"].copy()
rb_df = model_data[model_data["position"] == "RB"].copy()
wr_df = model_data[model_data["position"] == "WR"].copy()
te_df = model_data[model_data["position"] == "TE"].copy()